In [1]:
import pandas as pd

In [2]:
#ファイル読み込み
df = pd.read_csv('../000.data/train/train_A.tsv', sep="\t")
tdf = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 末尾が "_A" のテストデータだけを抽出
tdf_A = tdf[tdf["user_id"].str.endswith("_A")]

In [4]:
# 行動に対応する重みを定義
event_weights = {
    3: 3,  # 購入
    2: 2,  # 広告クリック
    1: 1,  # 詳細ページの閲覧
    0: 0 # カートに入れる
}

In [5]:
# スコア計算
def calculate_score(row):
    if row["event_type"] == 3 and row["ad"] != 1:
        return 0  # 購入は広告クリック(ad==1)時のみ有効
    return event_weights.get(row["event_type"], 0)

df["score"] = df.apply(calculate_score, axis=1)

In [6]:
# ユーザーごとに商品単位でスコアを合計
user_product_score = df.groupby(["user_id", "product_id"], as_index=False)["score"].sum()

In [7]:
# ユーザーごとにスコア降順、同スコアなら product_id 昇順で順位付け
user_product_score.sort_values(by=["user_id", "score", "product_id"], ascending=[True, False, True], inplace=True)
user_product_score["rank"] = user_product_score.groupby("user_id")["score"].rank(method="first", ascending=False).astype(int)

In [8]:
#前処理結果をcsvへ
user_product_score.to_csv('./train/train_A.csv', index=False)
tdf_A.to_csv('./test/test_A.csv', index=False)